[Back to NLP guideline](Natural-Language-Processing.html)


## **Modern NLP Systems**

A language model is a computational component. A useful NLP product is a system that surrounds that component with data, context construction, retrieval, tools, validation, permissions, observability, and human processes. The distinction matters because many real failures do not originate in the model weights. They originate in stale documents, malformed prompts, missing access controls, invalid tool arguments, broken retries, or an interface that encourages users to trust unsupported output.

This chapter follows one running example: a customer-support assistant that answers policy questions and, after confirmation, can create a support ticket. The example begins as a standalone model call and gradually acquires the components required for trustworthy operation:

```text
user request
-> input policy and identity checks
-> context construction
-> retrieval and reranking
-> grounded response generation
-> structured validation
-> optional tool execution
-> response and citations
-> trace, monitoring, and human escalation
```

The goal is not to promote maximum architectural complexity. Every component should exist because it addresses a concrete requirement. A deterministic workflow is often better than an autonomous agent, and a standalone classifier is often better than a generative system. Modern system design is the discipline of choosing the smallest architecture that can satisfy the task contract under real operational constraints.

### **From Model to System**

#### **Model, Pipeline, and System**

A **model** maps an input representation to an output. A classifier maps text to label scores; a language model maps a token prefix to a distribution over next tokens. The model does not know whether a user is authorized, whether a policy document is current, or what should happen after a prediction.

A **pipeline** composes several processing stages in a mostly fixed sequence. A classical support pipeline may detect language, classify intent, retrieve policy passages, and generate a response. Pipelines make intermediate data visible and allow each stage to be tested.

A **system** includes the pipeline plus external state and operational controls: document stores, indexes, APIs, permissions, caches, queues, versioning, user interfaces, monitoring, fallback behavior, and human ownership.

| Layer | Main Responsibility | Example Artifact | Typical Failure |
|---|---|---|---|
| model | statistical prediction or generation | model weights and tokenizer | wrong label or unsupported text |
| pipeline | ordered transformation of data | retrieval-generation graph | wrong stage ordering or schema mismatch |
| system | reliable behavior in an environment | service, index, tools, logs, policy | unauthorized action or undetected degradation |

The layers have different contracts. A generator may promise `text in -> text out`. The support system must promise something stronger: answers use the correct customer's permitted context, policy claims cite current documents, actions require confirmation, failures are recorded, and unresolved cases reach a person.

<details>
<summary>Python Defining Explicit Interfaces Between System Components</summary>

```python
from dataclasses import dataclass, field

@dataclass(frozen=True)
class UserRequest:
    request_id: str
    user_id: str
    message: str
    locale: str = "en-AU"

@dataclass(frozen=True)
class Evidence:
    document_id: str
    version: str
    text: str
    score: float

@dataclass
class SystemResponse:
    answer: str
    citations: list[str] = field(default_factory=list)
    action_status: str = "none"
    needs_human: bool = False

def validate_boundary(request: UserRequest) -> None:
    # Step 1: enforce basic input invariants at the service boundary.
    if not request.request_id or not request.user_id:
        raise ValueError("request and user identity are required")
    if not request.message.strip():
        raise ValueError("message cannot be empty")

# Step 2: downstream components exchange typed records rather than
# relying on an undocumented collection of dictionary keys.
request = UserRequest(
    request_id="req-1042",
    user_id="user-77",
    message="How long does a duplicate-charge refund take?",
)
validate_boundary(request)
```

</details>

Typed boundaries do not guarantee correctness, but they localize errors. If retrieval returns an evidence object without a version, the problem can be rejected before generation instead of becoming an uncited claim in the final response.

#### **Offline Construction and Online Inference**

Modern NLP systems usually contain two connected paths.

The **offline path** prepares assets that can be reused across requests:

```text
collect and approve documents
-> parse and normalize
-> split into chunks
-> attach metadata and permissions
-> compute sparse and dense representations
-> build indexes
-> evaluate and publish a version
```

The **online path** serves one user interaction:

```text
authenticate request
-> understand query
-> retrieve permitted candidates
-> rerank and construct context
-> call model or tool
-> validate response
-> return result and record trace
```

Separating these paths reduces latency and makes releases reproducible. Expensive document parsing and embedding should not run for every question. At the same time, an offline index cannot be treated as timeless. A new policy version, deleted customer record, or changed permission must propagate to the serving path.

| Concern | Offline Decision | Online Consequence |
|---|---|---|
| document version | which source is authoritative? | which claim may be cited? |
| chunking | how much context is stored together? | what evidence can be retrieved? |
| metadata | which fields and permissions are indexed? | which filters can be applied? |
| embedding model | how are documents represented? | query encoder must be compatible |
| release gate | which index passes evaluation? | active traffic receives that version |

An index update should be an explicit release, not an invisible mutation. Record its document snapshot, parser, chunking configuration, embedding model, and evaluation results. Otherwise, a response may change even though the language model version remains constant.

#### **Components and End-to-End Data Flow**

An end-to-end diagram is useful only if every arrow has a contract. For the support assistant, the flow can be divided into a **data plane**, which processes user requests, and a **control plane**, which manages versions, policy, monitoring, and releases.

```text
DATA PLANE
request -> gateway -> orchestrator -> retriever -> reranker -> model -> validator
                                  \-> approved tools -> external services

CONTROL PLANE
document ingestion -> index release -----------^
prompt/model registry -> deployment config ----^
evaluation suite -> release gate --------------^
traces and feedback -> monitoring -> response -^
```

The orchestrator coordinates components but should not become an untestable block containing every policy. Authorization belongs at tool and data boundaries. Validation belongs immediately after untrusted outputs. Version information belongs in every trace.

A useful design review asks for each edge:

1. What schema crosses this boundary?
2. Which component is allowed to produce it?
3. How is it validated?
4. What timeout or failure can occur?
5. What fallback follows?
6. Which version and trace identifiers are recorded?

The answer often reveals that the "NLP problem" is partly a distributed-systems problem. A high-quality model cannot compensate for a retriever timeout that silently returns an empty context or a retry that creates the same ticket twice.

### **Model Interface and Context Construction**

The model sees a serialized context, not the application as humans understand it. System instructions, retrieved text, conversation history, tool definitions, and the current request all compete for a limited context window. Context construction is therefore a form of programming: it defines the evidence and constraints under which the model predicts.

#### **Prompt Templates and Instruction Context**

A prompt template should separate trusted instructions from untrusted content. In the support assistant, at least four layers exist:

| Layer | Example | Trust Level |
|---|---|---|
| system policy | answer only from supplied policy excerpts | trusted and versioned |
| application context | locale, product, permitted tools | trusted structured data |
| retrieved content | policy chunks and metadata | externally sourced, treat as data |
| user content | current message and quoted text | untrusted input |

Delimiters and field names help the model interpret these layers, but delimiters are not a security boundary. A retrieved page can contain text that looks like an instruction. The application must still control which tools are available and validate every action outside the model.

Prompts should be treated as versioned artifacts with tests. A change that improves tone may reduce citation compliance or increase refusal. Record the prompt version in traces and evaluate it against normal, difficult, and adversarial cases before deployment.

<details>
<summary>Python Building a Prompt from Explicit Trust Boundaries</summary>

```python
from dataclasses import dataclass

@dataclass(frozen=True)
class ContextChunk:
    citation_id: str
    text: str

def build_support_prompt(user_message, chunks, locale="en-AU"):
    # Step 1: keep policy in code-controlled text, not user-controlled fields.
    instructions = (
        "You are a customer-support assistant. "
        "Use only the supplied evidence for policy claims. "
        "Cite each policy claim with [source_id]. "
        "If evidence is insufficient, say so and request human review."
    )

    # Step 2: serialize retrieved material as labeled evidence.
    evidence = "\n\n".join(
        f"SOURCE {chunk.citation_id}\n{chunk.text}"
        for chunk in chunks
    )

    # Step 3: mark the user message as data and keep it last.
    return f"""{instructions}

LOCALE: {locale}

<EVIDENCE>
{evidence}
</EVIDENCE>

<USER_MESSAGE>
{user_message}
</USER_MESSAGE>
"""

prompt = build_support_prompt(
    "How long does a duplicate-charge refund take?",
    [ContextChunk("refund-policy-v3#p4", "Refunds take 3-5 business days.")],
)
print(prompt)
```

</details>

The template makes provenance inspectable, but the system must still verify that every cited identifier was actually supplied and that the user is permitted to see the source.

#### **Context Windows and Token Budgets**

A model's context window limits the total tokens available for instructions, user input, retrieved evidence, conversation history, tool schemas, and generated output. If the window is $W$, a basic budget is

$$
B_{\text{evidence}} = W
- B_{\text{instructions}}
- B_{\text{history}}
- B_{\text{user}}
- B_{\text{tools}}
- B_{\text{output}}
- B_{\text{margin}}.
$$

$B_{\text{evidence}}$ is the space left for retrieved text. Each other $B$ term is the tokens reserved for a context component, and $B_{\text{margin}}$ absorbs tokenizer variation and unexpected metadata. A negative value means the request cannot fit under the current policy and must be shortened, summarized, split, or rejected.

A larger window does not remove selection problems. Irrelevant context can distract the model, repeated chunks waste budget, and essential evidence can be buried. Context quality usually matters more than filling every available token.

<details>
<summary>Python Allocating a Context Budget with Priority Rules</summary>

```python
def allocate_context(window, instructions, user, tools, output, margin, history):
    fixed = instructions + user + tools + output + margin
    remaining = window - fixed

    if remaining <= 0:
        raise ValueError("fixed prompt components exceed the context window")

    # Step 1: cap history so an old conversation cannot consume all evidence space.
    history_budget = min(history, int(remaining * 0.35))

    # Step 2: dedicate the rest to evidence, the source of factual grounding.
    evidence_budget = remaining - history_budget

    return {
        "instructions": instructions,
        "user": user,
        "tools": tools,
        "history": history_budget,
        "evidence": evidence_budget,
        "output": output,
        "margin": margin,
    }

budget = allocate_context(
    window=8192,
    instructions=650,
    user=180,
    tools=500,
    output=700,
    margin=300,
    history=3000,
)
print(budget)
print("total:", sum(budget.values()))
```

</details>

Token counting should use the deployed tokenizer, not word count. The budget policy should also define what is truncated first. Silent truncation of system policy or the newest user message is rarely acceptable.

#### **Structured Outputs and Validation**

Free-form text is appropriate for explanations. It is a poor interface for downstream code. When the model must choose an action, return entities, or produce a decision record, the output should follow a schema.

For example:

```json
{
  "answer": "Duplicate-charge refunds take 3-5 business days.",
  "citations": ["refund-policy-v3#p4"],
  "requested_action": null,
  "needs_human": false
}
```

Schema-constrained decoding can reduce syntax errors, but application validation is still required. The JSON may be valid while containing a nonexistent citation, an unauthorized action, or an order identifier belonging to another user.

Validation should proceed from syntax to semantics:

```text
parse JSON
-> check required fields and types
-> check allowed values
-> verify references against supplied context
-> enforce user and tool permissions
-> apply domain policy
-> accept, repair, retry, or escalate
```

<details>
<summary>Python Validating a Model Response Beyond JSON Syntax</summary>

```python
import json

ALLOWED_ACTIONS = {None, "create_ticket"}

def validate_model_output(raw_output, supplied_sources):
    # Step 1: parsing establishes syntax, not trust.
    data = json.loads(raw_output)

    # Step 2: enforce the expected shape and primitive types.
    required = {"answer", "citations", "requested_action", "needs_human"}
    missing = required - data.keys()
    if missing:
        raise ValueError(f"missing fields: {sorted(missing)}")
    if not isinstance(data["citations"], list):
        raise TypeError("citations must be a list")

    # Step 3: reject references the model was never given.
    unknown = set(data["citations"]) - set(supplied_sources)
    if unknown:
        raise ValueError(f"unknown citations: {sorted(unknown)}")

    # Step 4: actions are constrained by application policy.
    if data["requested_action"] not in ALLOWED_ACTIONS:
        raise ValueError("requested action is not allowed")

    return data

raw = json.dumps({
    "answer": "Refunds take 3-5 business days [refund-policy-v3#p4].",
    "citations": ["refund-policy-v3#p4"],
    "requested_action": None,
    "needs_human": False,
})

print(validate_model_output(raw, {"refund-policy-v3#p4"}))
```

</details>

Retries should be bounded and observable. Repeatedly asking the same model to repair invalid output can increase latency without fixing a semantic misunderstanding. After a small number of attempts, deterministic fallback or human review is safer.

#### **Conversation State and Short-Term Memory**

A full transcript is not the same as state. The transcript records what was said; state records what the application currently believes is relevant and valid.

For a support conversation, state may contain:

```text
authenticated user
active order
current intent
confirmed facts
missing fields
tool results
pending action
user confirmation
```

State should include provenance. A product identifier extracted with low confidence from an old turn should not be treated like an identifier returned by an authenticated account API. Values also need expiration and correction rules.

Conversation summarization can reduce token usage, but a generated summary is lossy and may introduce errors. Preserve critical structured fields separately, retain links to original turns, and rebuild summaries when source turns change.

| Memory Form | Lifetime | Good For | Main Risk |
|---|---|---|---|
| recent transcript | current context window | local coherence | consumes tokens quickly |
| structured dialogue state | current task | slots, confirmations, tool state | stale or incorrectly extracted values |
| conversation summary | long conversation | compressed background | omission or summary hallucination |
| durable user memory | across sessions | preferences with consent | privacy, outdated assumptions, over-personalization |

Durable memory should be opt-in, editable, minimal, and purpose-limited. Storing every conversation by default creates privacy and governance problems while often providing little task value.

### **Retrieval-Augmented Generation**

Retrieval-augmented generation (RAG) combines a retriever, which selects external evidence, with a generator, which produces an answer conditioned on that evidence. It gives the system a non-parametric memory that can be updated independently of model weights.

A simplified probabilistic view is

$$
p(y \mid x) = \sum_{z \in \mathcal{Z}_k(x)}
p_{\eta}(z \mid x)\,p_{\theta}(y \mid x,z).
$$

$x$ is the user query, $z$ is a retrieved document or passage, $\mathcal{Z}_k(x)$ is the top-$k$ candidate set, $p_{\eta}(z \mid x)$ is the retriever's probability or relevance score, and $p_{\theta}(y \mid x,z)$ is the generator's probability of the answer given the query and evidence. The sum represents uncertainty over which retrieved passage supports the answer.

The original RAG architecture combines a query encoder and document index with a sequence generator:

> ![Original retrieval-augmented generation architecture](assets/rag-original-architecture.png){width=95%}
>
> A query retrieves top documents from non-parametric memory, and the generator conditions on them to produce the output. Source: [Lewis et al., Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401).

In production, RAG is usually a modular pipeline rather than one end-to-end trained graph. Parsing, filtering, hybrid retrieval, reranking, context packing, citation validation, and monitoring become explicit components.

#### **When Does a System Need RAG?**

RAG is appropriate when answers depend on information that is private, frequently changing, too large to place in every prompt, or required to be cited. Examples include policy manuals, product catalogs, research collections, legal documents, and organization-specific knowledge.

RAG is not automatically necessary when:

- the task is a stable classification problem with sufficient labeled data;
- all required context already appears in the user input;
- a deterministic database query can produce the exact answer;
- retrieval errors would be more harmful than a curated fixed prompt;
- the system cannot legally or operationally expose the indexed material.

Fine-tuning and RAG solve different problems. Fine-tuning changes model behavior, style, or task competence. RAG supplies external evidence at inference time. Fine-tuning a model on yesterday's policy is an inefficient way to maintain today's policy.

#### **Document Ingestion, Chunking, and Metadata**

RAG quality begins before retrieval. The ingestion process must preserve content, structure, provenance, permissions, and versioning. A PDF parser that drops headings or joins table columns incorrectly changes what the retriever can learn.

Chunking divides documents into retrievable units. Very small chunks provide precise matches but may lose context. Very large chunks preserve context but dilute relevance and consume token budget. Structural boundaries such as sections, paragraphs, list items, or table rows are often better than fixed character counts.

Useful metadata includes:

| Field | Purpose |
|---|---|
| document and chunk ID | stable citation and deduplication |
| title and section path | context and display |
| source URI | provenance |
| version and effective date | freshness |
| language and domain | filtering and routing |
| access-control labels | permission enforcement |
| parser and embedding version | reproducibility |

<details>
<summary>Python Creating Overlapping Chunks with Provenance</summary>

```python
from dataclasses import dataclass

@dataclass(frozen=True)
class Chunk:
    chunk_id: str
    document_id: str
    version: str
    start_word: int
    text: str

def chunk_document(document_id, version, text, size=80, overlap=15):
    words = text.split()
    step = size - overlap
    if step <= 0:
        raise ValueError("overlap must be smaller than chunk size")

    chunks = []
    for start in range(0, len(words), step):
        piece = words[start:start + size]
        if not piece:
            break

        # Step 1: create a stable ID from source identity and position.
        chunk_id = f"{document_id}:{version}:w{start}"
        chunks.append(Chunk(
            chunk_id=chunk_id,
            document_id=document_id,
            version=version,
            start_word=start,
            text=" ".join(piece),
        ))

        # Step 2: stop after the final partial chunk.
        if start + size >= len(words):
            break

    return chunks

policy = "Duplicate charge refunds are processed within three to five business days. " * 20
for chunk in chunk_document("refund-policy", "v3", policy)[:2]:
    print(chunk.chunk_id, len(chunk.text.split()))
```

</details>

Overlap can preserve sentences crossing boundaries, but too much overlap fills the top results with near-duplicates. Deduplication and diversity-aware selection should be applied before context construction.

#### **Sparse, Dense, and Hybrid Retrieval**

Sparse retrieval represents text through lexical features and excels at exact terms, codes, names, and rare phrases. Dense retrieval represents semantic meaning through learned embeddings and can match paraphrases. Hybrid retrieval uses both because support queries frequently mix natural language with exact identifiers.

For example, `Why was AX-1042 charged again?` contains a semantic request about duplicate billing and an exact order-like identifier. Dense retrieval may understand the issue; sparse retrieval is more likely to preserve the identifier.

A practical retrieval pipeline is:

```text
metadata and permission filters
-> sparse candidate search
-> dense candidate search
-> rank fusion
-> deduplication
-> reranking
```

Filters should be applied as early as the retrieval engine permits. Retrieving a forbidden document and merely hiding its citation after generation is too late; its contents may already have influenced the answer.

#### **Reranking and Context Construction**

First-stage retrieval optimizes speed over a large collection. A reranker applies a more expensive model to a small candidate set and estimates relevance using the query and document together. Cross-encoders often provide better relevance judgments than independent embeddings because they can model token-level interactions.

Context construction then selects which reranked chunks fit the budget. Pure top-score selection can return redundant passages. A useful objective balances relevance, diversity, authority, recency, and token cost:

$$
U(d) = \lambda_r R(d) + \lambda_a A(d) + \lambda_f F(d)
- \lambda_n N(d,S) - \lambda_c C(d).
$$

$R(d)$ is relevance, $A(d)$ is source authority, $F(d)$ is freshness, $N(d,S)$ is redundancy with already selected set $S$, and $C(d)$ is token cost. The $\lambda$ values express product priorities. This is a design objective rather than a universal formula.

Context should preserve document identity and section boundaries. Merging chunks into an unlabeled wall of text makes citation and conflict resolution difficult. When two versions disagree, the system needs a version policy rather than asking the generator to guess.

#### **Grounded Generation and Citations**

A grounded answer makes claims supported by supplied evidence. A citation is useful only when it points to evidence that actually supports the nearby claim. Merely attaching the top retrieved source to the end of a response does not establish grounding.

Grounding controls can include:

- instructing the model to use only supplied sources for policy claims;
- requiring sentence- or claim-level citation identifiers;
- verifying that cited identifiers exist in the context;
- checking entailment between claims and cited passages;
- refusing or escalating when evidence is insufficient or contradictory;
- displaying excerpts so users can inspect the source.

<details>
<summary>Python Checking Citation Coverage and Source Validity</summary>

```python
import re

supplied_sources = {
    "refund-policy-v3#p4": "Refunds take three to five business days.",
    "billing-guide-v2#p8": "Duplicate charges may be disputed through support.",
}

answer = (
    "Duplicate-charge refunds take three to five business days "
    "[refund-policy-v3#p4]. Contact support to open a dispute "
    "[billing-guide-v2#p8]."
)

def validate_citations(answer, supplied_sources):
    citations = re.findall(r"\[([^\]]+)\]", answer)

    # Step 1: every citation must refer to evidence actually supplied.
    unknown = [item for item in citations if item not in supplied_sources]

    # Step 2: split prose into claims and check that each has a citation.
    claims = [part.strip() for part in re.split(r"(?<=[.!?])\s+", answer) if part.strip()]
    uncited = [claim for claim in claims if not re.search(r"\[[^\]]+\]", claim)]

    return {
        "citations": citations,
        "unknown_citations": unknown,
        "uncited_claims": uncited,
    }

report = validate_citations(answer, supplied_sources)
print(report)
assert not report["unknown_citations"]
assert not report["uncited_claims"]
```

</details>

This validator checks format and coverage, not semantic support. Claim-evidence verification requires an NLI model, a specialized evaluator, or human review. High-impact claims may need deterministic extraction from authoritative fields rather than free-form generation.

#### **RAG Evaluation and Failure Modes**

RAG must be evaluated at multiple layers:

| Layer | Key Question | Example Metric or Test |
|---|---|---|
| ingestion | was the source parsed and versioned correctly? | parse coverage and table tests |
| retrieval | did the candidate set contain supporting evidence? | recall@k |
| ranking | did useful evidence appear early? | MRR or nDCG |
| context | was the selected evidence relevant and non-redundant? | context precision and token use |
| generation | is the answer correct and complete? | task rubric or answer accuracy |
| grounding | are claims supported by cited evidence? | claim support and citation precision |
| system | does the complete workflow help users reliably? | resolution rate, latency, escalation |

Common failure chains include:

```text
bad parse -> missing terms -> retrieval miss -> unsupported answer
stale index -> obsolete evidence -> confidently outdated answer
duplicate chunks -> narrow context -> omitted exception
weak permission filter -> private evidence -> information disclosure
good evidence + weak prompt -> uncited synthesis
good answer + invalid citation -> user cannot verify it
```

The diagnostic order should follow the pipeline. Before changing the generator, check whether the correct evidence was available, retrieved, reranked, and included.

### **Tool-Using and Agentic Workflows**

Retrieval gives a model information. Tools let a system observe or change an external environment. A tool may query an order database, calculate a value, create a ticket, send a message, or schedule an appointment. Because tool effects can outlive the conversation, tool use requires stronger controls than text generation.

#### **Function Calling**

Function calling asks the model to produce a structured proposal containing a tool name and arguments. The application, not the model, decides whether and how to execute it.

```json
{
  "tool": "create_support_ticket",
  "arguments": {
    "category": "duplicate_charge",
    "order_id": "A-1042"
  }
}
```

The tool schema constrains syntax and helps the model select arguments. It is not authorization. Before execution, the application must authenticate the user, verify that the order belongs to the user, check required confirmation, and enforce allowed values.

<details>
<summary>Python Dispatching an Allowlisted Tool with Policy Checks</summary>

```python
from dataclasses import dataclass

@dataclass(frozen=True)
class ToolRequest:
    name: str
    arguments: dict

def create_support_ticket(category, order_id):
    # A real implementation would call an idempotent external API.
    return {"ticket_id": "T-9001", "category": category, "order_id": order_id}

TOOLS = {"create_support_ticket": create_support_ticket}
ALLOWED_CATEGORIES = {"duplicate_charge", "refund", "delivery"}

def dispatch_tool(request, user_order_ids, confirmed):
    # Step 1: allow only registered tools.
    if request.name not in TOOLS:
        raise PermissionError("tool is not allowlisted")

    # Step 2: validate arguments against domain and user permissions.
    category = request.arguments.get("category")
    order_id = request.arguments.get("order_id")
    if category not in ALLOWED_CATEGORIES:
        raise ValueError("invalid ticket category")
    if order_id not in user_order_ids:
        raise PermissionError("order is not accessible to this user")

    # Step 3: require explicit confirmation for an external side effect.
    if not confirmed:
        return {"status": "confirmation_required"}

    return TOOLS[request.name](category=category, order_id=order_id)

proposal = ToolRequest(
    name="create_support_ticket",
    arguments={"category": "duplicate_charge", "order_id": "A-1042"},
)
print(dispatch_tool(proposal, {"A-1042"}, confirmed=True))
```

</details>

Side-effecting tools should support idempotency keys so retries do not create duplicate tickets or payments. Read-only and write tools should have separate permissions and clearer user confirmation requirements.

#### **Reason, Act, and Observe**

Tool-using systems often operate as a loop:

```text
inspect current state
-> choose an allowed action
-> execute through the application
-> receive a structured observation
-> update state
-> stop, continue, or escalate
```

ReAct demonstrated the value of interleaving reasoning and actions with observations from an external environment. In a production system, the important engineering artifact is the observable action trajectory. Private internal reasoning need not be exposed or stored; a concise action rationale and structured state are usually more useful for audit and recovery.

> ![ReAct examples comparing reasoning-only, action-only, and interleaved reasoning and acting](assets/react-reason-act-observe.png){width=92%}
>
> The examples show that reasoning without new observations can propagate an early mistake, while action without sufficient reasoning can select the wrong next step. Source: [Yao et al., ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629).

The loop must have stopping conditions. Maximum steps, time budget, cost budget, repeated-action detection, and terminal success states prevent indefinite execution. An agent that repeatedly searches the same query is not becoming more certain; it is consuming resources without changing state.

#### **State and Memory**

An agentic workflow needs a state object that survives between model calls. State should distinguish:

| State Type | Example | Storage Rule |
|---|---|---|
| task state | ticket category and missing fields | retain for current workflow |
| observations | order API result | record source and timestamp |
| plan or pending steps | verify order, then request confirmation | revise when observations change |
| action history | tool, arguments, result | append to trace |
| durable memory | user prefers email follow-up | store only with purpose and consent |

Do not use the model's generated prose as the only state store. Structured state supports validation, deterministic transitions, and resumption after failure. It also prevents an old model message from being mistaken for a current tool result.

#### **Permissions and Error Recovery**

The principle of least privilege gives each workflow only the tools, documents, and operations required for the current task. A policy-answering assistant does not need a payment tool. A ticket-creation workflow may need write access only after identity and confirmation checks.

Tool errors should be classified rather than blindly retried:

| Error | Example | Response |
|---|---|---|
| validation | missing order ID | ask user or repair arguments |
| authorization | order belongs to another account | deny and log security event |
| transient | service timeout | bounded retry with backoff |
| rate limit | too many requests | wait, queue, or use fallback |
| permanent domain error | order is ineligible | explain result and alternatives |
| ambiguous result | tool may have succeeded before timeout | check idempotency status before retry |

Recovery should return the workflow to a known state. If a tool partially succeeds, compensation or human intervention may be needed. The model should not invent success because a tool failed to respond.

#### **Workflow vs Agent**

A **workflow** follows a predefined graph of steps and conditions. An **agent** chooses among actions dynamically based on observations. Many systems combine both: a deterministic workflow controls authorization and high-impact transitions, while a model chooses how to search or explain within bounded steps.

| Property | Deterministic Workflow | Agentic Loop |
|---|---|---|
| path | predefined | selected at runtime |
| predictability | high | lower |
| flexibility | limited to known branches | adapts to unfamiliar paths |
| testing | enumerate transitions | evaluate trajectories and budgets |
| best use | stable business process | open-ended information gathering |
| main risk | brittle missing branch | uncontrolled action or looping |

Use an agent only when dynamic action selection creates real value. If a support ticket always requires `collect order -> verify eligibility -> confirm -> submit`, a workflow expresses the requirement more clearly and reliably.

### **Production and Observability**

A prototype demonstrates that one request can succeed. Production engineering asks whether the service remains useful under load, failures, version changes, adversarial inputs, and cost limits. Observability connects an outcome to the exact model, prompt, index, tools, and decisions that produced it.

#### **Latency, Throughput, and Cost**

End-to-end latency is approximately the sum of sequential stages plus queueing:

$$
T_{\text{total}} = T_{\text{gateway}} + T_{\text{retrieval}}
+ T_{\text{rerank}} + T_{\text{generation}} + T_{\text{tools}}
+ T_{\text{validation}} + T_{\text{queue}}.
$$

Each term is elapsed time for one component. Components executed in parallel contribute their maximum rather than their sum. For example, sparse and dense retrieval can run concurrently.

Average latency hides poor user experience in the tail. Services normally monitor percentiles such as p50, p95, and p99. If p95 is 8 seconds, one in twenty requests takes at least that long even if the average is much lower.

Throughput is completed work per unit time, such as requests per second or generated tokens per second. Cost includes model input and output tokens, embedding, reranking, storage, network calls, tools, and human review:

$$
C_{\text{request}} = C_{\text{model}} + C_{\text{retrieval}}
+ C_{\text{tools}} + C_{\text{infrastructure}} + C_{\text{review}}.
$$

Optimization should preserve quality. Reducing retrieved evidence may cut tokens but increase unsupported answers and human escalations, raising total cost.

#### **Caching, Batching, and Model Routing**

Caching avoids repeated work when inputs and versions match. Useful caches include document embeddings, retrieval results for stable public queries, and deterministic model responses. Cache keys must include every factor that changes the result, such as model, prompt, index, locale, permissions, and decoding configuration.

Never share a cached response across users if it contains private context. Permission-aware retrieval results require user or access-scope information in the key, which may reduce cache reuse but preserves isolation.

Batching groups compatible requests to improve hardware utilization. Dynamic batching waits briefly to combine requests, creating a latency-throughput tradeoff. Streaming reduces perceived generation latency by returning tokens incrementally, although validation and tool decisions may require buffering structured output.

Model routing chooses a model based on task, risk, complexity, language, or budget. A small classifier can handle intent routing, a compact generator can answer simple grounded questions, and a stronger model can receive ambiguous or multi-document cases. Routing itself must be evaluated because sending a difficult case to an inadequate model creates hidden quality loss.

#### **Tracing and Versioning**

A trace records the path of one request across components. Logs answer local questions; traces connect distributed events into one causal timeline.

At minimum, record:

```text
request and trace IDs
timestamps and duration per stage
model and tokenizer version
prompt and policy version
index and document versions
retrieved and selected chunk IDs
tool proposals, approvals, and results
validation and fallback decisions
token use, cost estimate, and final status
```

Sensitive text should be minimized, redacted, encrypted, access controlled, and retained only as needed. Observability must not become an uncontrolled copy of user conversations.

<details>
<summary>Python Recording Structured Trace Events for One Request</summary>

```python
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
import json

@dataclass(frozen=True)
class TraceEvent:
    trace_id: str
    stage: str
    status: str
    version: str
    duration_ms: int
    details: dict
    timestamp: str

def event(trace_id, stage, status, version, duration_ms, **details):
    # Step 1: create one versioned event without storing raw private text.
    return TraceEvent(
        trace_id=trace_id,
        stage=stage,
        status=status,
        version=version,
        duration_ms=duration_ms,
        details=details,
        timestamp=datetime.now(timezone.utc).isoformat(),
    )

trace = [
    event("tr-1042", "retrieval", "ok", "index-2026-07-16", 43,
          selected_chunk_ids=["refund-policy:v3:w160"]),
    event("tr-1042", "generation", "ok", "model-4.2+prompt-18", 812,
          input_tokens=980, output_tokens=96),
    event("tr-1042", "citation_validation", "ok", "validator-3", 4,
          citation_count=1),
]

# Step 2: emit structured records that monitoring and audit tools can query.
for item in trace:
    print(json.dumps(asdict(item)))
```

</details>

Version combinations matter. A model may behave correctly with prompt 18 and index 42 but fail with prompt 19. Reproducing the output requires the complete configuration, not just the model name.

#### **Monitoring, Fallback, and Human Escalation**

Monitoring should cover service health, model behavior, retrieval, safety, cost, and user outcomes.

| Signal | Example Alert |
|---|---|
| availability | error rate exceeds service objective |
| latency | p95 generation latency doubles |
| retrieval | no-result rate rises after index release |
| grounding | unsupported-claim sample rate exceeds threshold |
| tools | repeated or unauthorized action attempts increase |
| cost | tokens per resolved case rise sharply |
| quality | human correction or reopening rate increases |
| drift | new language or product slice grows |

A metric needs an owner and response. When retrieval fails, fallback might show search results or route to a person. When the generator times out, a deterministic acknowledgement may be better than retrying indefinitely. When evidence conflicts, the system should expose the conflict rather than manufacture certainty.

Human escalation is part of the architecture, not evidence that automation failed. Define triggers such as low confidence, missing evidence, policy conflict, high-impact action, repeated tool failure, explicit user request, or detected distress. The reviewer needs the relevant evidence and trace, not just the final model answer.

Deployment should be gradual. Offline evaluation, shadow traffic, canary release, and rollback reduce blast radius. Monitor both average behavior and critical slices after every model, prompt, index, or tool change.

### **Choosing a System Architecture**

Architecture selection begins with requirements, not components. Ask whether the system needs external knowledge, whether knowledge changes, whether an action must occur, whether the action path is known, and how serious a wrong answer or action would be.

#### **Standalone Model vs RAG vs Tool-Using System**

| Architecture | Adds | Best Fit | Main New Failure Mode |
|---|---|---|---|
| standalone model | model call and validation | stable classification, rewriting, supplied-context tasks | model error or unsupported generation |
| model plus fixed context | curated instructions and examples | narrow stable policy | stale or oversized prompt |
| RAG | external index, retrieval, citations | changing or private knowledge | retrieval and grounding failure |
| deterministic tool workflow | validated API actions | stable business process | integration and state-transition failure |
| bounded agent | dynamic tool selection and observations | open-ended multi-step investigation | looping, wrong action, and cost growth |

Complexity should be earned. If the answer is available through an exact database field, query it directly. If users need documents, provide search. If they need a synthesized answer with evidence, add RAG. If they need an action, add a constrained tool workflow. Add agentic planning only when the path cannot be represented effectively in advance.

#### **Architecture Decision Matrix**

| Requirement | Model Only | RAG | Workflow + Tools | Bounded Agent |
|---|---:|---:|---:|---:|
| classify or rewrite supplied text | strong fit | unnecessary | unnecessary | unnecessary |
| answer from changing documents | weak fit | strong fit | optional | optional |
| provide auditable citations | weak fit | strong fit | optional | possible but complex |
| execute a known transaction | cannot act | cannot act alone | strong fit | often excessive |
| investigate through unknown steps | limited | retrieves only | brittle | strong fit |
| predictable latency and behavior | strongest | moderate | moderate to strong | weakest |
| easiest component testing | strongest | moderate | moderate | hardest |

Risk modifies the decision. A low-impact research assistant may tolerate dynamic exploration. A financial or clinical action should place deterministic validation and human authorization around any model-selected step.

#### **End-to-End NLP System Case Study**

The support assistant can now be assembled as a bounded architecture:

```text
1. authenticate user and create trace
2. classify request and select workflow
3. retrieve permitted policy chunks
4. rerank, deduplicate, and fit context budget
5. generate a structured answer with citations
6. validate schema, citations, and policy
7. if an action is requested, gather missing fields
8. ask for explicit confirmation
9. execute one allowlisted idempotent tool
10. return result or escalate with complete trace
```

<details>
<summary>Python Sketching the Complete Grounded Support Workflow</summary>

```python
def handle_support_request(request, services):
    trace_id = services.tracing.start(request.request_id)

    try:
        # Step 1: establish identity and the resources this user may access.
        identity = services.auth.authenticate(request.user_id)
        access_scope = services.auth.policy_scope(identity)

        # Step 2: retrieve first, then verify that sufficient evidence exists.
        candidates = services.retrieval.search(
            query=request.message,
            access_scope=access_scope,
            index_version="policy-index-42",
        )
        evidence = services.reranker.select(candidates, token_budget=2400)
        if not evidence:
            return services.handoff.create(
                trace_id=trace_id,
                reason="no_authoritative_evidence",
            )

        # Step 3: generate a structured proposal, not an executable side effect.
        prompt = services.prompts.build(
            template_version="support-grounded-18",
            request=request,
            evidence=evidence,
        )
        raw_output = services.model.generate(
            prompt,
            model_version="support-model-4.2",
        )

        # Step 4: validate citations and requested actions against real context.
        response = services.validation.validate(
            raw_output,
            supplied_sources={item.chunk_id for item in evidence},
            allowed_actions={None, "create_support_ticket"},
        )

        # Step 5: keep write actions behind deterministic confirmation policy.
        if response.requested_action == "create_support_ticket":
            if not request.confirmed:
                return services.responses.request_confirmation(response)

            tool_result = services.tools.execute(
                name="create_support_ticket",
                arguments=response.action_arguments,
                identity=identity,
                idempotency_key=request.request_id,
            )
            response.action_status = tool_result.status

        services.tracing.finish(trace_id, status="ok")
        return response

    except services.transient_errors as error:
        # Step 6: known transient failures use a bounded fallback path.
        services.tracing.finish(trace_id, status="fallback", error=type(error).__name__)
        return services.responses.safe_fallback(
            "The support service is temporarily unavailable. No action was taken."
        )
    except Exception as error:
        # Step 7: unknown failures preserve evidence and escalate.
        services.tracing.finish(trace_id, status="error", error=type(error).__name__)
        return services.handoff.create(trace_id=trace_id, reason="system_error")
```

</details>

The example is intentionally modular. Each service can be replaced, tested, versioned, and observed independently. The model is important, but it is not allowed to authenticate users, invent sources, or execute arbitrary functions.

A practical architecture review can end with six checks:

1. **Necessity:** does every component solve a stated requirement?
2. **Contracts:** are inputs, outputs, versions, and failure states explicit?
3. **Evidence:** can factual claims be traced to permitted sources?
4. **Authority:** are model proposals separated from application authorization?
5. **Recovery:** do timeouts, invalid outputs, and partial actions reach known states?
6. **Observability:** can the team reconstruct, monitor, and improve the complete path?

Modern NLP systems are not defined by how many model calls they contain. They are defined by how deliberately statistical language capabilities are connected to evidence, software, people, and real consequences.